# Parcel OCR — run the whole pipeline

This notebook drives the repo end to end, the same way the on-prem scheduler will:

1. **Data**: generate synthetic parcel jobs (rendered Thai labels, rosters, review-service export,
   two-reviewer gold labels), ingest them, and freeze the regression and acceptance sets
2. **Cycle 1**: `parcel-ocr retrain` versions the data (DVC), trains (PaddleOCR Thai recognizer +
   type/size classifier), calibrates thresholds, assembles a bundle, evaluates it and runs the
   gate, then registers the challenger (MLflow)
3. **Promote** the challenger to champion with a named approver, then run inference with it
4. **Cycle 2**: new data, retrain, gate **vs the champion**, promote
5. **Rollback** to the previous champion, then inspect lineage (git tags, `dvc.lock`, MLflow versions)

**Where to run it**

| Environment | How |
|---|---|
| Docker (on-prem) | `make notebook`, then open http://127.0.0.1:8888 and enter `NOTEBOOK_TOKEN` |
| Google Colab | Runtime → GPU (T4), open this notebook, run all. It clones the repo from GitHub |

Everything happens in an **isolated workspace**: a copy of the repo with its own git history, a
local DVC remote folder, and a local MLflow database (SQLite). Nothing touches your real `data/`,
the SeaweedFS bucket or the MLflow server.

> Synthetic data proves the pipeline **runs**. Its accuracy numbers say nothing about real parcels.
> Never put real parcel images on Colab: label images must stay inside the organization.

Time on a Colab T4: about 15–25 minutes for everything. CPU only: hours (lower `N_JOBS_*`).

In [ ]:
# 1 · Settings
N_JOBS_BATCH1 = 1200  # synthetic jobs for cycle 1 (25% become the frozen evaluation sets)
N_JOBS_BATCH2 = 600  # new jobs arriving before cycle 2
DEMO_EPOCHS = 3  # training epochs per model (real runs: see params.yaml)
FRESH = False  # True wipes the workspace and starts again (needed to pick up code changes)
USE_LOCAL_MLFLOW = True  # False uses MLFLOW_TRACKING_URI from the environment (MLflow server)
REPO_URL = "https://github.com/ntuankiet1/OCR.git"

In [ ]:
# 2 · Locate the repo (clone it on Colab) and create the isolated workspace
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_repo() -> Path | None:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "cli.py").exists() and (p / "dvc.yaml").exists():
            return p
    return None


SOURCE = find_repo()
if SOURCE is None:
    SOURCE = Path("/content/OCR" if IN_COLAB else "OCR").resolve()
    if not SOURCE.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(SOURCE)], check=True)

WORKSPACE = Path("/content/parcel-ocr-demo") if IN_COLAB else SOURCE / ".workspace" / "demo"
DVC_REMOTE = WORKSPACE.parent / f"{WORKSPACE.name}-dvc-remote"
TOP_LEVEL_SKIP = {".git", ".workspace", "data", "artifacts", "reports", "mlruns", ".venv", ".env"}


def ignore(dirpath: str, names: list[str]) -> set[str]:
    """Copy code and config only: no data, artifacts, caches or client documents."""
    skip = {n for n in names if n in {"__pycache__", ".pytest_cache", ".ruff_cache"}}
    skip |= {n for n in names if n.endswith(".pdf")}
    if Path(dirpath) == SOURCE:
        skip |= TOP_LEVEL_SKIP & set(names)
    if Path(dirpath) == SOURCE / ".dvc":
        skip |= {"cache", "tmp", "config.local"} & set(names)
    return skip


def git(*args: str) -> None:
    subprocess.run(["git", *args], cwd=WORKSPACE, check=True)


if FRESH:
    shutil.rmtree(WORKSPACE, ignore_errors=True)
    shutil.rmtree(DVC_REMOTE, ignore_errors=True)
if not WORKSPACE.exists():
    shutil.copytree(SOURCE, WORKSPACE, ignore=ignore)
    (WORKSPACE / "data" / "frozen").mkdir(parents=True, exist_ok=True)
    with open(WORKSPACE / ".gitignore", "a") as f:
        f.write("\n# demo workspace\nmlflow.db\nmlruns/\n.bin/\n")
    git("init", "-q", "-b", "main")
    git("config", "user.name", "parcel-ocr-demo")  # local identity: Colab has none
    git("config", "user.email", "demo@localhost")
    git("add", "-A")
    git("commit", "-q", "-m", "workspace: copy of the repo")
print("source:   ", SOURCE)
print("workspace:", WORKSPACE)

In [ ]:
# 3 · Dependencies: present in the Docker image; installed on the fly on Colab / elsewhere
import importlib.util
import tomllib

IN_DOCKER = Path("/opt/PaddleOCR/tools/train.py").exists()
PADDLE_VERSION = "3.2.2"  # 3.3.x breaks CPU/oneDNN inference (Paddle #77340)
PADDLE_GPU_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
HAS_GPU = (
    bool(shutil.which("nvidia-smi"))
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
pip = [sys.executable, "-m", "pip", "install", "-q"]


def installed(dist: str) -> str | None:
    from importlib.metadata import PackageNotFoundError, version

    try:
        return version(dist)
    except PackageNotFoundError:
        return None


if not IN_DOCKER:
    if IN_COLAB and not Path("/usr/share/fonts/opentype/tlwg").exists():
        subprocess.run(
            "apt-get -qq update && apt-get -qq install -y fonts-thai-tlwg libfribidi0",
            shell=True,
            check=True,
        )
    # PaddlePaddle: the CPU wheel never uses a GPU. Use the CUDA wheel when a GPU is visible.
    want, other = (
        ("paddlepaddle-gpu", "paddlepaddle") if HAS_GPU else ("paddlepaddle", "paddlepaddle-gpu")
    )
    if installed(want) != PADDLE_VERSION or installed(other):
        already_loaded = "paddle" in sys.modules
        if installed(other):
            subprocess.run(
                [sys.executable, "-m", "pip", "uninstall", "-y", "-q", other], check=True
            )
        extra = ["-i", PADDLE_GPU_INDEX] if HAS_GPU else []
        subprocess.run([*pip, f"{want}=={PADDLE_VERSION}", *extra], check=True)
        if already_loaded:
            raise RuntimeError(
                "paddle changed in a running kernel: restart the runtime, then run all"
            )
    core = tomllib.loads((WORKSPACE / "pyproject.toml").read_text())["project"]["dependencies"]
    training = ["paddleocr==3.7.0", "timm", "onnx", "onnxruntime", "onnxscript"]
    if importlib.util.find_spec("torch") is None:
        training += ["torch", "torchvision"]
    subprocess.run([*pip, *core, *training], check=True)
    PADDLEOCR_HOME = WORKSPACE.parent / "PaddleOCR"
    if not (PADDLEOCR_HOME / "tools" / "train.py").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "--depth",
                "1",
                "--branch",
                "v3.7.0",
                "https://github.com/PaddlePaddle/PaddleOCR",
                str(PADDLEOCR_HOME),
            ],
            check=True,
        )
    subprocess.run([*pip, "-r", str(PADDLEOCR_HOME / "requirements.txt")], check=True)
else:
    PADDLEOCR_HOME = Path("/opt/PaddleOCR")

import nltk  # noqa: E402
import paddle  # noqa: E402

for resource, package in (("corpora/wordnet", "wordnet"), ("corpora/omw-1.4", "omw-1.4")):
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True)
GPU = paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
print(
    f"paddle {paddle.__version__} · GPU: {GPU} · "
    f"python {sys.version.split()[0]} · docker: {IN_DOCKER}"
)

In [ ]:
# 4 · Environment + helpers. All commands run inside the workspace.
import pandas as pd
from IPython.display import HTML, display

BIN = WORKSPACE / ".bin"
BIN.mkdir(exist_ok=True)
shim = BIN / "parcel-ocr"  # DVC stages call `parcel-ocr`: run this workspace's code
shim.write_text(f'#!/bin/sh\nexec "{sys.executable}" -m src.cli "$@"\n')
shim.chmod(0o755)

os.environ["PATH"] = f"{BIN}:{os.environ['PATH']}"
os.environ["PARCEL_OCR_PADDLEOCR_HOME"] = str(PADDLEOCR_HOME)
if USE_LOCAL_MLFLOW:
    os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{WORKSPACE}/mlflow.db"
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command in the workspace and stream its output."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, cwd=WORKSPACE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed ({rc}): {cmd}")
    return rc


def show_gate() -> dict:
    gate = json.loads((WORKSPACE / "reports" / "gate.json").read_text())
    m = json.loads((WORKSPACE / "reports" / "challenger" / "metrics.json").read_text())
    verdict = "PASSED" if gate["passed"] else "FAILED"
    display(HTML(f"<h3>Gate {verdict} · {gate['mode']}</h3>"))
    display(pd.DataFrame(gate["checks"]))
    print(
        f"auto rate {m['auto_rate']:.1%} · wrong assignments {m['wrong_assignments']} · "
        f"whole-record accuracy {m['whole_record_accuracy']:.1%} · "
        f"latency p50/p95 {m['latency_ms']['p50']:.0f}/{m['latency_ms']['p95']:.0f} ms"
    )
    display(pd.DataFrame(m["fields"]).T[["n", "accuracy", "ci_low", "ci_high"]].round(4))
    return gate


print("mlflow:", os.environ.get("MLFLOW_TRACKING_URI"))

## Demo parameters
Synthetic data is small and the demo hardware is not the reference server, so a few
`params.yaml` values are relaxed **in the workspace copy only**:

- fewer epochs
- calibration at 90% precision with at least 30 accepted samples (real: 99.5% and 200)
- a latency ceiling suited to a notebook
- wider gate tolerances

The hard rule, **zero wrong assignments**, is not relaxed.

In [ ]:
# 5 · Demo parameters (workspace copy only) + local DVC remote
import yaml

params_path = WORKSPACE / "params.yaml"
params = yaml.safe_load(params_path.read_text())
params["train_rec"].update(epochs=DEMO_EPOCHS, warmup_epochs=1, use_gpu=bool(GPU), num_workers=2)
params["train_type_size"].update(epochs=DEMO_EPOCHS, num_workers=2)
params["calibration"].update(target_precision=0.90, min_accepted=30)
params["gate"].update(
    max_latency_p95_ms=60000,
    max_field_drop_pp=2.0,
    max_subgroup_drop_pp=5.0,
    max_auto_rate_drop_pp=2.0,
    subgroup_min_n=30,
)
params_path.write_text(yaml.safe_dump(params, sort_keys=False, allow_unicode=True))
sh("git add params.yaml && git diff --cached --quiet || git commit -q -m 'demo parameters'")

DVC_REMOTE.mkdir(parents=True, exist_ok=True)
sh(f"dvc remote add --local -d -f demo {DVC_REMOTE}")  # .dvc/config.local, not committed
sh("dvc remote list --local")

## Data

In [ ]:
# 6 · Data directories + pretrained base models (downloaded once, then versioned with DVC)
sh("parcel-ocr infra init-dirs")
if not (WORKSPACE / "artifacts" / "base" / "det").exists():
    sh("parcel-ocr fetch-base-models")
sh("dvc add artifacts/base")

In [ ]:
# 7 · Synthetic batch 1: label photos, review export, gold labels, rosters
import cv2
import matplotlib.pyplot as plt

sh(f"parcel-ocr demo-data --n {N_JOBS_BATCH1} --batch 1")
B1 = WORKSPACE / "data" / "demo" / "batch-1"
events = [json.loads(x) for x in (B1 / "review_events.jsonl").read_text().splitlines()[:3]]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
rows = []
for ax, e in zip(axes, events, strict=True):
    ax.imshow(cv2.cvtColor(cv2.imread(e["image_uri"]), cv2.COLOR_BGR2RGB))
    ax.set_title(e["job_id"])
    ax.axis("off")
    rows.append(
        {
            "job": e["job_id"],
            "decided_by": e["decided_by"],
            **{
                k: v["value"]
                for k, v in e["final"].items()
                if k in ("recipient_name", "unit", "tracking_number", "carrier", "type_size")
            },
        }
    )
plt.show()
display(pd.DataFrame(rows))

In [ ]:
# 8 · Ingest (auto-registered jobs give no labels), gold labels, freeze evaluation sets
sh(f"parcel-ocr ingest {B1 / 'review_events.jsonl'}")
sh(f"parcel-ocr ingest-labels {B1 / 'gold_labels.jsonl'}")
for split in ("regression", "acceptance"):
    if not (WORKSPACE / "data" / "frozen" / f"{split}.txt").exists():
        sh(
            f"parcel-ocr freeze {split} --keys-file {B1 / f'frozen_{split}_keys.txt'} "
            f"--note 'synthetic demo, batch 1'"
        )
sh("git add data/frozen && git diff --cached --quiet || git commit -q -m 'freeze evaluation sets'")

## Cycle 1: first model
`retrain` runs `dvc add` on the raw data, re-scores the champion (there is none yet), runs `dvc repro`
(snapshot → export → train → calibrate → bundle → evaluate → gate), registers the bundle as
`challenger` if the gate passes, and pushes data and artifacts to the DVC remote.

In [ ]:
# 9 · Cycle 1
sh("parcel-ocr retrain --force")
stats = json.loads((WORKSPACE / "data" / "snapshots" / "current" / "stats.json").read_text())
display(pd.DataFrame({"images per split": stats["images_per_split"]}))
display(pd.DataFrame({"raw labels per source": stats["labels_per_source"]}))
gate1 = show_gate()

In [ ]:
# 10 · Commit the lineage: dvc.lock + .dvc pointers pin the exact data, params and models
sh("git add -A && git commit -q -m 'cycle 1: data snapshot + pipeline lock' && git tag -f cycle-1")
sh("dvc metrics show")

In [ ]:
# 11 · Registry: promote the challenger (after shadow mode, by a named approver)
def status() -> dict:
    from src.registry.mlflow_registry import status as registry_status

    s = registry_status("parcel-ocr-bundle")
    table = {a: (s.get(a) or {}) for a in ("champion", "challenger", "previous")}
    keys = ["version", "bundle_version", "approved_by", "rolled_back_by"]
    display(pd.DataFrame({a: {k: v.get(k) for k in keys} for a, v in table.items()}).T)
    print("champion history:", s.get("history"))
    return s


s = status()
if s.get("challenger"):
    sh(
        f"parcel-ocr promote {s['challenger']['version']} --approved-by demo-lead "
        "--note 'demo: shadow period skipped'"
    )
    s = status()
else:
    print("no challenger: the gate did not pass, so nothing can be promoted")

In [ ]:
# 12 · Inference with the champion: decision, review reasons and bundle id (audit, BR16)
from src.evaluation.predict import load_roster
from src.inference.pipeline import InferencePipeline
from src.registry.mlflow_registry import CHAMPION, download_bundle

bundle = download_bundle("parcel-ocr-bundle", CHAMPION, WORKSPACE / "artifacts" / "demo-champion")
if bundle is None:
    print("no champion yet")
else:
    pipe = InferencePipeline.from_bundle(bundle)
    imgs = pd.read_parquet(WORKSPACE / "data" / "snapshots" / "current" / "images.parquet")
    out = []
    for r in imgs[imgs["split"] == "regression"].head(5).itertuples():
        res = pipe.run(r.uri, load_roster(WORKSPACE / "data" / "rosters" / f"{r.property_id}.json"))
        out.append(
            {
                "image": r.image_sha256[:10],
                "decision": res.decision,
                "reasons": ", ".join(res.reasons),
                "unit": res.fields["unit"]["value"],
                "recipient": res.fields["recipient_name"]["value"],
                "tracking": res.fields["tracking_number"]["value"],
                "ms": round(res.latency_ms),
                "bundle": res.bundle_id[:8],
            }
        )
    display(pd.DataFrame(out))
    del pipe  # free GPU memory before cycle 2 trains

## Cycle 2: new data, gate against the champion
The gate now compares the challenger with the champion on the same frozen regression set:
per-field McNemar test, subgroup drops, auto rate, and zero wrong assignments.

In [ ]:
# 13 · Cycle 2
sh(f"parcel-ocr demo-data --n {N_JOBS_BATCH2} --batch 2")
sh(f"parcel-ocr ingest {WORKSPACE / 'data' / 'demo' / 'batch-2' / 'review_events.jsonl'}")
sh("parcel-ocr retrain --force")
gate2 = show_gate()
sh("git add -A && git commit -q -m 'cycle 2: data snapshot + pipeline lock' && git tag -f cycle-2")
s = status()
if s.get("challenger"):
    sh(
        f"parcel-ocr promote {s['challenger']['version']} --approved-by demo-lead "
        "--note 'demo: cycle 2'"
    )
    s = status()

## Rollback
Suppose the pilot finds wrong unit matches with the new champion. Rollback moves the `champion`
alias back to the previous version: serving picks it up with no redeploy, and the bad version is
tagged with who rolled it back and why.

In [ ]:
# 14 · Rollback
if len(s.get("history", [])) >= 2:
    sh("parcel-ocr rollback --reason 'demo: pilot found wrong unit matches' --by demo-ops")
    s = status()
else:
    print("only one champion so far: nothing to roll back to")

In [ ]:
# 15 · Lineage: which data, params and code produced which model
from mlflow import MlflowClient

sh("git log --oneline --decorate -n 8")
sh("dvc dag")
versions = MlflowClient().search_model_versions("name='parcel-ocr-bundle'")
display(
    pd.DataFrame(
        [
            {
                "version": v.version,
                "aliases": ", ".join(v.aliases),
                **{
                    k: v.tags.get(k)
                    for k in (
                        "bundle_version",
                        "gate_passed",
                        "approved_by",
                        "rolled_back_by",
                        "rollback_reason",
                    )
                },
            }
            for v in sorted(versions, key=lambda v: int(v.version))
        ]
    )
)
print("DVC remote content:", sum(1 for _ in DVC_REMOTE.rglob("*") if _.is_file()), "files")

## Going back in time
Any cycle can be reproduced exactly from its git tag:

```bash
cd <workspace>
git checkout cycle-1 && dvc checkout   # data, exports and artifacts of cycle 1
dvc repro                              # same inputs → same outputs
```

## Running on real data (on-prem only)
1. Export the review queue as JSONL ([`ReviewEvent`](../src/schemas.py)) → `parcel-ocr ingest`
2. Export Label Studio adjudications as JSONL (`LabelRecord`, source `adjudicated`) →
   `parcel-ocr ingest-labels`
3. Put resident rosters in `data/rosters/<property>.json`
4. Freeze the regression and acceptance sets once (`parcel-ocr freeze`)
5. Use the real `params.yaml` values and the MLflow server (`USE_LOCAL_MLFLOW = False`), and
   schedule `parcel-ocr retrain`; promotion stays manual